In [1]:
# set the directory
import os
path = '/result-3DHPE/base/mmPose-NLP'
os.chdir(path)
print(os.getcwd())
!ls

/result-3DHPE/base/mmPose-NLP
Dataset  result


In [5]:
import numpy as np
from sklearn import metrics
import pandas as pd

GT = np.load('result/GT_voxel.npy')
pred = np.load('result/pred_voxel.npy')
n_frames = GT.shape[0]
print('frames: ', n_frames)
# print(np.max(GT), np.min(GT))
# print(np.max(pred), np.min(pred))

voxel_dict = np.load('Dataset/VD.npy')
GroundTruth = np.zeros((np.shape(GT)[0],np.shape(GT)[1],3))
Prediction = np.zeros((np.shape(GT)[0],np.shape(GT)[1],3))

for i in range(np.shape(GT)[0]):
    for j in range(np.shape(GT)[1]):
        GroundTruth[i,j,:] = voxel_dict[int(GT[i,j])]
        Prediction[i,j,:] = voxel_dict[int(pred[i,j])]


gtpath = "result/GT_cont_test.npy" 
predpath = "result/Pred_cont_test.npy" 

# np.save(gtpath,GroundTruth)
# np.save(predpath,Prediction)
# print("--------------------------------")
# print(GroundTruth.shape)  #(7984, 19, 3)
# print(GroundTruth[:2])
# print(Prediction.shape)
# print(Prediction[:2])
# print("--------------------------------")

# mean_3d_error = np.mean(abs(GroundTruth-Prediction),axis=0)
# print(mean_3d_error)
# print(mean_3d_error.shape)

# mean_3d_error = np.mean(mean_3d_error, axis=0)
# print(mean_3d_error)
# print(mean_3d_error.shape)

# ---------------- MAE 计算 ----------------
# 分别计算 x, y, z 方向的 MAE（每个轴19个关节的所有帧误差）
# 先转置，将最后两个轴互换，使得 shape 变为 (N, 3, 19)
GroundTruth_transposed = GroundTruth.transpose(0, 2, 1)
Prediction_transposed = Prediction.transpose(0, 2, 1)
# 再 reshape 成 (N, 57)，这样前 19 列为 x 坐标，中间 19 列为 y 坐标，最后 19 列为 z 坐标
labels_test = GroundTruth_transposed.reshape(GroundTruth.shape[0], -1)
predictions = Prediction_transposed.reshape(Prediction.shape[0], -1)

x_mae = metrics.mean_absolute_error(labels_test[:, 0:19], predictions[:, 0:19], multioutput='raw_values')
y_mae = metrics.mean_absolute_error(labels_test[:, 19:38], predictions[:, 19:38], multioutput='raw_values')
z_mae = metrics.mean_absolute_error(labels_test[:, 38:57], predictions[:, 38:57], multioutput='raw_values')

# 将三个轴的误差拼接成一个 3 x 19 的矩阵，然后转置为 19 x 3
all_19_points_mae = np.concatenate((x_mae, y_mae, z_mae)).reshape(3, 19)
all_19_points_mae_transpose = all_19_points_mae.T
print("MAE for 19 joints (each row corresponds to one joint, columns for x, y, z):")
print(all_19_points_mae_transpose)

# 计算各轴平均 MAE（沿 0 轴取均值），为一个 1x3 的结果
avg_19_points_mae_xyz = np.mean(all_19_points_mae, axis=1).reshape(1, 3)
print("Average MAE for each axis (x, y, z):")
# print(avg_19_points_mae_xyz)

# 分别输出三个轴的总体 MAE（所有关节综合的误差）
mae_x_total = metrics.mean_absolute_error(labels_test[:, 0:19], predictions[:, 0:19])
mae_y_total = metrics.mean_absolute_error(labels_test[:, 19:38], predictions[:, 19:38])
mae_z_total = metrics.mean_absolute_error(labels_test[:, 38:57], predictions[:, 38:57])
print("MAE for x is", mae_x_total)
print("MAE for y is", mae_y_total)
print("MAE for z is", mae_z_total)

# ---------------- 新指标 ----------------

# 1. 计算19个关节每个关节误差（x、y、z误差取平均），得到 19 维向量
avg_joint_error = np.mean(all_19_points_mae_transpose, axis=1)  # shape (19,)
print("Average error for each joint (x, y, z average):")
print(avg_joint_error)

# 2. 计算总体 RMSE（均方根误差）：分别计算 x, y, z 轴所有19个关节的 RMSE
rmse_x = np.sqrt(metrics.mean_squared_error(labels_test[:, 0:19], predictions[:, 0:19]))
rmse_y = np.sqrt(metrics.mean_squared_error(labels_test[:, 19:38], predictions[:, 19:38]))
rmse_z = np.sqrt(metrics.mean_squared_error(labels_test[:, 38:57], predictions[:, 38:57]))
print("Overall RMSE for x is", rmse_x)
print("Overall RMSE for y is", rmse_y)
print("Overall RMSE for z is", rmse_z)

# 3. 计算每个关节每个轴的 RMSE（19x3矩阵）
joint_rmse_x = np.sqrt(np.mean((labels_test[:, 0:19] - predictions[:, 0:19]) ** 2, axis=0))
joint_rmse_y = np.sqrt(np.mean((labels_test[:, 19:38] - predictions[:, 19:38]) ** 2, axis=0))
joint_rmse_z = np.sqrt(np.mean((labels_test[:, 38:57] - predictions[:, 38:57]) ** 2, axis=0))
# 组合为 (19, 3) 形状的矩阵，其中每行依次对应：关节i的RMSE_x, RMSE_y, RMSE_z
joint_rmse_matrix = np.vstack([joint_rmse_x, joint_rmse_y, joint_rmse_z]).T
print("Per-joint RMSE for x, y, z (shape: 19x3):")
print(joint_rmse_matrix)

# ---------------- 保存结果到同一个 Excel 文件（单 Sheet） ----------------

# 构造汇总表（Summary）：使用单个 DataFrame 保存指标，每行一项指标
# 对于向量（如 avg_19_points_mae_xyz 和 avg_joint_error），以逗号分隔的字符串形式保存
summary_data = {
    "Metric": [
        "MAE_x (total)",
        "MAE_y (total)",
        "MAE_z (total)",
        "Average MAE for each axis (x,y,z)",  # 平均 MAE 的 1x3 向量
        "Average error for each joint (x,y,z avg) - 19 values",
        "Overall RMSE_x",
        "Overall RMSE_y",
        "Overall RMSE_z"
    ],
    "Value": [
        mae_x_total,
        mae_y_total,
        mae_z_total,
        ", ".join(["{:.4f}".format(v) for v in avg_19_points_mae_xyz.flatten()]),
        ", ".join(["{:.4f}".format(v) for v in avg_joint_error]),
        rmse_x,
        rmse_y,
        rmse_z
    ]
}
df_summary = pd.DataFrame(summary_data)

# 构造 19*3 的 Joint_RMSE 表，增加关节编号
df_joint_rmse = pd.DataFrame(joint_rmse_matrix, columns=['RMSE_x', 'RMSE_y', 'RMSE_z'])
df_joint_rmse.insert(0, 'Joint Index', range(1, 20))  # 关节编号从 1 到 19

# 将两个表合并写入单个工作表中：先写入汇总表，再空一行，再写入关节 RMSE 表
excel_output_path = 'result/mmPose-NLP_result.xlsx'
with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
    # 写入汇总表
    df_summary.to_excel(writer, sheet_name='Result', index=False, startrow=0)
    
    # 计算汇总表写入后的行数，预留一行空白
    start_row = df_summary.shape[0] + 2
    # 写入 Joint_RMSE 表
    df_joint_rmse.to_excel(writer, sheet_name='Result', index=False, startrow=start_row)

print("All results have been saved to:", excel_output_path)


frames:  7984
MAE for 19 joints (each row corresponds to one joint, columns for x, y, z):
[[0.05191884 0.03348697 0.06215932]
 [0.05571142 0.02641784 0.07183367]
 [0.06198397 0.02937876 0.07637275]
 [0.06960421 0.03396794 0.08498497]
 [0.0665982  0.03281563 0.07459419]
 [0.08206413 0.04345691 0.09757515]
 [0.13577154 0.05713427 0.16379259]
 [0.06286072 0.03400802 0.07504509]
 [0.08747495 0.04445892 0.09642285]
 [0.12932866 0.0528507  0.17192385]
 [0.05267034 0.03493487 0.06223447]
 [0.05869739 0.03809619 0.03705912]
 [0.06627756 0.03800601 0.02577154]
 [0.07440882 0.04598196 0.03083166]
 [0.05281563 0.03480461 0.06325651]
 [0.06142285 0.04406814 0.04210421]
 [0.07409319 0.04068637 0.03105711]
 [0.08146794 0.04250501 0.04371242]
 [0.06106713 0.02906814 0.07504008]]
Average MAE for each axis (x, y, z):
MAE for x is 0.07295986710262764
MAE for y is 0.03874353971100181
MAE for z is 0.07293534437295703
Average error for each joint (x, y, z average):
[0.04918838 0.05132098 0.05591182 0.06285